In [1]:
!git clone https://github.com/face-analysis/emonet.git
!cd emonet
!pip install -r requirements.txt

Cloning into 'emonet'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 132 (delta 55), reused 49 (delta 49), pack-reused 66 (from 1)
Receiving objects: 100% (132/132), 100.25 MiB | 14.18 MiB/s, done.
Resolving deltas: 100% (60/60), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
import torch
from emonet.emonet import models

In [3]:

# 1. Configuration
n_expression = 8  # Use 5 or 8 depending on the model you want
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 2. Instantiate the model
net = models.EmoNet(n_expression=n_expression).to(device)

# 3. Load pre-trained weights
# The weights are located in the 'pretrained' folder of the repo
weights_path = f'emonet/pretrained/emonet_{n_expression}.pth'
state_dict = torch.load(weights_path, map_location='cpu')

# Clean up state dictionary (remove 'module.' prefix if present)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

# Load weights into the model
net.load_state_dict(state_dict, strict=False)
net.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [5]:
import cv2
import numpy as np
import torch

# ... (Assume 'net' is loaded from the previous step) ...

def preprocess_image(image_path):
    # Load image using OpenCV
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Resize to 256x256 (Model requirement)
    image = cv2.resize(image, (256, 256))
    
    # Normalize to [0, 1] and permute dimensions to (Channels, Height, Width)
    image_tensor = torch.Tensor(image).permute(2, 0, 1) / 255.0
    
    # Add batch dimension: (1, 3, 256, 256)
    return image_tensor.unsqueeze(0).to(device)

# Run Inference
image_path = "/kaggle/input/human-face-emotions/data/Happy/2560px-Happy_People.jpg"  # Replace with your image
input_tensor = preprocess_image(image_path)

with torch.no_grad():
    output = net(input_tensor)

# 4. Parse Output
# The model returns a dictionary with keys: 'expression', 'valence', 'arousal'
expression_logits = output['expression']
valence = output['valence'].clamp(-1.0, 1.0).item()
arousal = output['arousal'].clamp(-1.0, 1.0).item()

# Get the predicted class index
predicted_class = torch.argmax(torch.nn.functional.softmax(expression_logits, dim=1)).item()

emotion_labels = {
    0: "Neutral", 1: "Happy", 2: "Sad", 3: "Surprise", 
    4: "Fear", 5: "Disgust", 6: "Anger", 7: "Contempt"
}

print(f"Emotion: {emotion_labels[predicted_class]}")
print(f"Valence: {valence:.3f}")
print(f"Arousal: {arousal:.3f}")

Emotion: Neutral
Valence: 0.016
Arousal: 0.256


In [ ]:
torch.nn.functional.softmax(expression_logits, dim=1).cpu().numpy()

## TESTING

In [9]:
!pip install face-recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 18.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=fa6c4eb2d22d74bf8984232fe66387250f26798eaa0da23262ce9d6eb1905187
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [14]:
import os
import sys
import torch
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_EXPRESSION = 8
DATASET_PATH = "/kaggle/input/human-face-emotions/data"

# Mapping from your dataset folder names to EmoNet's class indices
# EmoNet: {0:Neutral, 1:Happy, 2:Sad, 3:Surprise, 4:Fear, 5:Disgust, 6:Anger, 7:Contempt}
CLASS_MAP = {
    "Angry": 6,
    "Happy": 1,
    "Sad": 2
}

# -----------------------------------------------------------------------------
# 2. LOAD MODEL
# -----------------------------------------------------------------------------
print(f"Loading EmoNet (n={N_EXPRESSION}) on {DEVICE}...")
net = models.EmoNet(n_expression=N_EXPRESSION).to(DEVICE)

# Load weights
weights_path = f'emonet/pretrained/emonet_{N_EXPRESSION}.pth'
if not os.path.exists(weights_path):
    print(f"Error: Weights not found at {weights_path}")
    # You might need to adjust this path depending on where you run the script
    sys.exit(1)

state_dict = torch.load(weights_path, map_location=DEVICE)
# Remove 'module.' prefix if it exists (common in DataParallel models)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
net.load_state_dict(state_dict, strict=False)
net.eval()

# -----------------------------------------------------------------------------
# 3. PREPROCESSING WITH FACE DETECTION
# -----------------------------------------------------------------------------
# Try to import face_recognition, fall back to Haar Cascades if missing
try:
    import face_recognition
    USE_FACE_RECOGNITION = True
    print("Using 'face_recognition' library for detection.")
except ImportError:
    USE_FACE_RECOGNITION = False
    print("Warning: 'face_recognition' not found. Falling back to OpenCV Haar Cascades.")
    # Load Haar Cascade
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def preprocess_image(image_path):
    """
    Loads image, DETECTS FACE, crops it, resizes to 256x256, and normalizes.
    Returns: Tensor (1, 3, 256, 256) or None if no face found.
    """
    try:
        if USE_FACE_RECOGNITION:
            # Load file
            image_rgb = face_recognition.load_image_file(image_path)
            
            # Detect faces (returns list of top, right, bottom, left)
            face_locations = face_recognition.face_locations(image_rgb)
            
            if not face_locations:
                return None
            
            # Use the first face found
            top, right, bottom, left = face_locations[0]
            
            # Expand the box slightly (optional, EmoNet likes tight crops but a little margin helps)
            margin = 0
            h, w, _ = image_rgb.shape
            top = max(0, top - margin)
            bottom = min(h, bottom + margin)
            left = max(0, left - margin)
            right = min(w, right + margin)
            
            face_image = image_rgb[top:bottom, left:right]
            
        else:
            # OpenCV Fallback
            image = cv2.imread(image_path)
            if image is None: return None
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)
            if len(faces) == 0:
                return None
            
            x, y, w, h = faces[0]
            face_image = image_rgb[y:y+h, x:x+w]

        # Resize to 256x256
        face_image = cv2.resize(face_image, (256, 256))
        
        # Normalize [0,1] and transpose to (C, H, W)
        face_tensor = torch.tensor(face_image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        
        # Add batch dimension
        return face_tensor.unsqueeze(0).to(DEVICE)

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# -----------------------------------------------------------------------------
# 4. EVALUATION LOOP
# -----------------------------------------------------------------------------
gt_labels = []
pred_labels = []
faces_not_found = 0

print(f"\nStarting evaluation on {DATASET_PATH}...")

with torch.no_grad():
    for label_name, label_idx in CLASS_MAP.items():
        folder_path = os.path.join(DATASET_PATH, label_name)
        
        # Find all images
        extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        image_files = []
        for ext in extensions:
            image_files.extend(glob(os.path.join(folder_path, ext)))
            image_files.extend(glob(os.path.join(folder_path, ext.upper())))
            
        print(f"Processing class '{label_name}' (mapped to {label_idx}). Found {len(image_files)} images.")
        
        for img_path in tqdm(image_files):
            input_tensor = preprocess_image(img_path)
            
            if input_tensor is None:
                faces_not_found += 1
                continue
            
            # Run Inference
            output = net(input_tensor)
            
            # Output is a dict: {'expression': (B, 8), 'valence': (B, 1), 'arousal': (B, 1), ...}
            expression_logits = output['expression']
            
            # Get prediction
            pred_idx = torch.argmax(expression_logits, dim=1).item()
            
            # Store results
            gt_labels.append(label_idx)
            pred_labels.append(pred_idx)

# -----------------------------------------------------------------------------
# 5. RESULTS
# -----------------------------------------------------------------------------
if len(gt_labels) == 0:
    print("No images processed successfully.")
else:
    acc = accuracy_score(gt_labels, pred_labels)
    
    print("\n" + "="*40)
    print(" PYTORCH MODEL RESULTS")
    print("="*40)
    print(f"Images Processed: {len(gt_labels)}")
    print(f"Faces Not Found:  {faces_not_found} (skipped)")
    print(f"Accuracy:         {acc:.4f} ({acc*100:.2f}%)")
    print("="*40)

Loading EmoNet (n=8) on cuda...
Using 'face_recognition' library for detection.

Starting evaluation on /kaggle/input/human-face-emotions/data...
Processing class 'Angry' (mapped to 6). Found 86 images.


100%|██████████| 86/86 [01:07<00:00,  1.27it/s]


Processing class 'Happy' (mapped to 1). Found 100 images.


100%|██████████| 100/100 [01:30<00:00,  1.11it/s]


Processing class 'Sad' (mapped to 2). Found 78 images.


100%|██████████| 78/78 [01:12<00:00,  1.08it/s]


 PYTORCH MODEL RESULTS
Images Processed: 159
Faces Not Found:  105 (skipped)
Accuracy:         0.4025 (40.25%)


In [17]:
import os
import sys
import torch
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# # Ensure we can import from the cloned repository
# if os.path.isdir('emonet'):
#     sys.path.append('emonet')

# from emonet.models import EmoNet

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_EXPRESSION = 8
DATASET_ROOT = "/kaggle/input/facial-emotion-recognition/images"

# Mapping from Filename (without .jpg) to EmoNet class index
# EmoNet: {0:Neutral, 1:Happy, 2:Sad, 3:Surprise, 4:Fear, 5:Disgust, 6:Anger, 7:Contempt}
FILENAME_MAP = {
    "Neutral": 0,
    "Happy": 1,
    "Sad": 2,
    "Surprised": 3,
    "Fear": 4,
    "Disgust": 5,
    "Anger": 6,
    "Contempt": 7
}

# -----------------------------------------------------------------------------
# 2. LOAD MODEL
# -----------------------------------------------------------------------------
print(f"Loading EmoNet (n={N_EXPRESSION}) on {DEVICE}...")
net = models.EmoNet(n_expression=N_EXPRESSION).to(DEVICE)

weights_path = f'emonet/pretrained/emonet_{N_EXPRESSION}.pth'
if not os.path.exists(weights_path):
    print(f"Error: Weights not found at {weights_path}")
    sys.exit(1)

state_dict = torch.load(weights_path, map_location=DEVICE)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
net.load_state_dict(state_dict, strict=False)
net.eval()

# -----------------------------------------------------------------------------
# 3. PREPROCESSING (CENTER CROP)
# -----------------------------------------------------------------------------
def preprocess_center_crop(image_path):
    """
    Loads image, performs Center Crop (Square), Resizes to 256x256, Normalizes.
    """
    try:
        # Load image
        image = cv2.imread(image_path)
        if image is None: return None
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        h, w, _ = image.shape
        
        # Center Crop Calculation
        # We take the smallest dimension to be the side of the square
        min_dim = min(h, w)
        
        top = (h - min_dim) // 2
        left = (w - min_dim) // 2
        
        # Crop
        image_cropped = image[top : top+min_dim, left : left+min_dim]
        
        # Resize to 256x256
        image_resized = cv2.resize(image_cropped, (256, 256))
        
        # Normalize [0,1] and transpose to (C, H, W)
        image_tensor = torch.tensor(image_resized, dtype=torch.float32).permute(2, 0, 1) / 255.0
        
        return image_tensor.unsqueeze(0).to(DEVICE)

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# -----------------------------------------------------------------------------
# 4. EVALUATION LOOP
# -----------------------------------------------------------------------------
gt_labels = []
pred_labels = []
missing_files = 0

# Folders are named "0", "1", ... "18"
folders = [str(i) for i in range(19)] 

print(f"\nStarting evaluation on {DATASET_ROOT}...")

with torch.no_grad():
    for folder_name in tqdm(folders, desc="Processing Folders"):
        folder_path = os.path.join(DATASET_ROOT, folder_name)
        
        if not os.path.isdir(folder_path):
            continue
            
        # Iterate through the expected filenames (Neutral.jpg, Happy.jpg, etc.)
        for emotion_name, label_idx in FILENAME_MAP.items():
            filename = f"{emotion_name}.jpg"
            image_path = os.path.join(folder_path, filename)
            
            if not os.path.exists(image_path):
                # Some folders might not have all emotions
                missing_files += 1
                continue
            
            input_tensor = preprocess_center_crop(image_path)
            
            if input_tensor is None:
                continue
                
            # Run Inference
            output = net(input_tensor)
            expression_logits = output['expression']
            pred_idx = torch.argmax(expression_logits, dim=1).item()
            
            # Store results
            gt_labels.append(label_idx)
            pred_labels.append(pred_idx)

# -----------------------------------------------------------------------------
# 5. RESULTS
# -----------------------------------------------------------------------------
if len(gt_labels) == 0:
    print("No images processed successfully.")
else:
    acc = accuracy_score(gt_labels, pred_labels)
    
    print("\n" + "="*40)
    print(" DATASET RESULTS (CENTER CROP)")
    print("="*40)
    print(f"Total Images Evaluated: {len(gt_labels)}")
    print(f"Missing/Skipped Files:  {missing_files}")
    print(f"Accuracy:               {acc:.4f} ({acc*100:.2f}%)")
    print("="*40)

Loading EmoNet (n=8) on cuda...

Starting evaluation on /kaggle/input/facial-emotion-recognition/images...


Processing Folders: 100%|██████████| 19/19 [00:13<00:00,  1.43it/s]


 DATASET RESULTS (CENTER CROP)
Total Images Evaluated: 152
Missing/Skipped Files:  0
Accuracy:               0.5724 (57.24%)


## ONNX

In [18]:
!pip install protobuf==3.20.3
!pip install onnxruntime -q
!pip install onnx -q
!pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
a2a-sdk 0.3.22 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have pr

In [19]:
import torch

# 1. Define a Wrapper to handle Dictionary Output
class EmonetWrapper(torch.nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.model = original_model

    def forward(self, x):
        out = self.model(x)
        # Return as tuple: (heatmap, expression, valence, arousal)
        # Note: valence and arousal are scalars in the dict, we keep them that way or unsqueeze if needed
        return out['heatmap'], out['expression'], out['valence'], out['arousal']

# 2. Load the Original Model
n_expression = 8
device = 'cpu'  # Exporting on CPU is often safer/simpler
net = models.EmoNet(n_expression=n_expression).to(device)

weights_path = f'emonet/pretrained/emonet_{n_expression}.pth'
state_dict = torch.load(weights_path, map_location=device)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
net.load_state_dict(state_dict, strict=False)
net.eval()

# 3. Wrap the model
wrapped_net = EmonetWrapper(net)

# wrapped_net.eval()
# heatmap, expression, valence, arousal = wrapped_net(dummy_input)

# expression.shape
# # torch.Size([1, 8])
# valence.shape
# arousal.shape
# # torch.Size([1])


In [20]:

# 4. Create dummy input (Batch, Channel, Height, Width)
dummy_input = torch.randn(1, 3, 256, 256).to(device)

# 5. Export
output_onnx_path = f"emonet_{n_expression}.onnx"
torch.onnx.export(
    wrapped_net,
    dummy_input,
    output_onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['heatmap', 'expression', 'valence', 'arousal'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'heatmap': {0: 'batch_size'},
        'expression': {0: 'batch_size'},
        'valence': {0: 'batch_size'},
        'arousal': {0: 'batch_size'}
    }
)

print(f"Model exported to {output_onnx_path}")

/tmp/ipykernel_55/2666644232.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


Model exported to emonet_8.onnx


In [21]:
from onnxruntime.quantization import quantize_dynamic, QuantType

model_fp32 = "emonet_8.onnx"
model_quant = "emonet_8_int8.onnx"

print(f"Quantizing {model_fp32}...")

quantize_dynamic(
    model_input=model_fp32,
    model_output=model_quant,
    weight_type=QuantType.QUInt8  # Quantize weights to 8-bit integers
)

print(f"Quantized model saved to {model_quant}")

Quantizing emonet_8.onnx...
Quantized model saved to emonet_8_int8.onnx


In [22]:
import os
import cv2
import numpy as np
import onnxruntime as ort
from glob import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# 1. Metric Implementations (Based on Emonet repo)
def PCC(ground_truth, predictions):
    """Pearson Correlation Coefficient"""
    return np.corrcoef(ground_truth, predictions)[0, 1]

def CCC(ground_truth, predictions):
    """Concordance Correlation Coefficient"""
    mean_pred = np.mean(predictions)
    mean_gt = np.mean(ground_truth)
    std_pred = np.std(predictions)
    std_gt = np.std(ground_truth)
    pearson = PCC(ground_truth, predictions)
    return 2.0 * pearson * std_pred * std_gt / (std_pred**2 + std_gt**2 + (mean_pred - mean_gt)**2)

def MSE(ground_truth, predictions):
    """Mean Squared Error"""
    return np.mean((ground_truth - predictions)**2)

# 2. Setup
dataset_path = "/kaggle/input/human-face-emotions/data"
# Emonet Mapping: {0:Neutral, 1:Happy, 2:Sad, 3:Surprise, 4:Fear, 5:Disgust, 6:Anger, 7:Contempt}
# Dataset Folders: Angry, Happy, Sad
class_map = {
    "Angry": 6,
    "Happy": 1,
    "Sad": 2
}

def preprocess_image(image_path):
    try:
        image = cv2.imread(image_path)
        if image is None: return None
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))
        # Normalize to [0, 1] and CHW layout
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        return np.expand_dims(image, axis=0)
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None

# 3. Load Models
print("Loading ONNX sessions...")
session_fp32 = ort.InferenceSession("emonet_8.onnx")
session_int8 = ort.InferenceSession("emonet_8_int8.onnx")

# 4. Evaluation Loop
gt_emotions = []
fp32_preds = {'emo': [], 'val': [], 'aro': []}
int8_preds = {'emo': [], 'val': [], 'aro': []}

print("Running inference...")
for label_name, label_idx in class_map.items():
    folder_path = os.path.join(dataset_path, label_name)
    image_files = glob(os.path.join(folder_path, "*.*")) # Matches .jpg, .png etc
    
    print(f"Processing class {label_name} ({len(image_files)} images)...")
    
    for img_path in tqdm(image_files):
        input_tensor = preprocess_image(img_path)
        if input_tensor is None: continue
        
        # FP32 Inference
        outs_fp32 = session_fp32.run(None, {"input": input_tensor})
        # Unpack: heatmap, expression, valence, arousal (Indices match export script)
        emo_logits_fp32, val_fp32, aro_fp32 = outs_fp32[1], outs_fp32[2], outs_fp32[3]
        
        # INT8 Inference
        outs_int8 = session_int8.run(None, {"input": input_tensor})
        emo_logits_int8, val_int8, aro_int8 = outs_int8[1], outs_int8[2], outs_int8[3]
        
        # Store Results
        gt_emotions.append(label_idx)
        
        fp32_preds['emo'].append(np.argmax(emo_logits_fp32))
        fp32_preds['val'].append(val_fp32.item())
        fp32_preds['aro'].append(aro_fp32.item())
        
        int8_preds['emo'].append(np.argmax(emo_logits_int8))
        int8_preds['val'].append(val_int8.item())
        int8_preds['aro'].append(aro_int8.item())

# 5. Calculate Metrics

# Convert to numpy arrays
gt_emotions = np.array(gt_emotions)
for k in fp32_preds: fp32_preds[k] = np.array(fp32_preds[k])
for k in int8_preds: int8_preds[k] = np.array(int8_preds[k])

print("\n" + "="*40)
print(" COMPARISON RESULTS")
print("="*40)

# A. Accuracy (Ground Truth vs Predictions)
acc_fp32 = accuracy_score(gt_emotions, fp32_preds['emo'])
acc_int8 = accuracy_score(gt_emotions, int8_preds['emo'])

print(f"Emotion Classification Accuracy:")
print(f"  FP32 (Original):  {acc_fp32:.4f}")
print(f"  INT8 (Quantized): {acc_int8:.4f}")
print(f"  Delta:            {acc_int8 - acc_fp32:.4f}")

# B. Quantization Fidelity (FP32 vs INT8)
# Since we lack Ground Truth for V/A, we measure how much INT8 deviates from FP32
mse_val = MSE(fp32_preds['val'], int8_preds['val'])
mse_aro = MSE(fp32_preds['aro'], int8_preds['aro'])
ccc_val = CCC(fp32_preds['val'], int8_preds['val'])
ccc_aro = CCC(fp32_preds['aro'], int8_preds['aro'])

print("\nQuantization Fidelity (FP32 vs INT8):")
print("(Measures how close the quantized model is to the original)")
print(f"  Valence MSE: {mse_val:.6f} (Lower is better)")
print(f"  Arousal MSE: {mse_aro:.6f} (Lower is better)")
print(f"  Valence CCC: {ccc_val:.6f} (Higher is better, max 1.0)")
print(f"  Arousal CCC: {ccc_aro:.6f} (Higher is better, max 1.0)")
print("="*40)

Loading ONNX sessions...
Running inference...
Processing class Angry (86 images)...


 29%|██▉       | 25/86 [00:12<00:30,  1.98it/s]


KeyboardInterrupt: 

Compare speed

In [ ]:

import time
import numpy as np
import onnxruntime as ort

def benchmark_session(model_path, inputs, n_warmup=10):
    print(f"Loading {model_path}...")
    options = ort.SessionOptions()
    # Optional: Graph optimization level can be set here
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    session = ort.InferenceSession(model_path, options)
    input_name = session.get_inputs()[0].name
    
    # Warmup (Crucial to initialize internal allocators)
    print("  Warming up...")
    for _ in range(n_warmup):
        session.run(None, {input_name: inputs[0]})
        
    # Benchmark
    print(f"  Running inference on {len(inputs)} samples...")
    start_time = time.time()
    for img in inputs:
        session.run(None, {input_name: img})
    end_time = time.time()
    
    total_time = end_time - start_time
    avg_time_ms = (total_time / len(inputs)) * 1000
    fps = len(inputs) / total_time
    
    return avg_time_ms, fps

def main():
    # 1. Prepare Data
    count = 100
    print(f"Generating {count} dummy inputs (1, 3, 256, 256)...")
    # Random float32 data to simulate normalized images
    dummy_inputs = [np.random.randn(1, 3, 256, 256).astype(np.float32) for _ in range(count)]

    print("-" * 40)

    # 2. Benchmark FP32
    fp32_ms, fp32_fps = benchmark_session("emonet_8.onnx", dummy_inputs)
    print(f"FP32 Result: {fp32_ms:.2f} ms/sample  |  {fp32_fps:.2f} FPS")

    print("-" * 40)

    # 3. Benchmark INT8
    int8_ms, int8_fps = benchmark_session("emonet_8_int8.onnx", dummy_inputs)
    print(f"INT8 Result: {int8_ms:.2f} ms/sample  |  {int8_fps:.2f} FPS")

    print("=" * 40)
    
    # 4. Results
    speedup = fp32_fps / int8_fps
    # Or equivalently: fp32_ms / int8_ms
    
    print(f"Speedup Factor: {speedup:.2f}x")
    if speedup > 1:
        print(f"The quantized model is {speedup:.2f} times faster.")
    else:
        print("Note: If speedup is ~1.0x or less, your CPU might not support VNNI instructions,")
        print("or the overhead of the small batch size dominates the computation.")

main()

In [23]:
import os
import cv2
import numpy as np
import onnxruntime as ort
from glob import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# -----------------------------------------------------------------------------
# 1. METRIC IMPLEMENTATIONS
# -----------------------------------------------------------------------------
def PCC(ground_truth, predictions):
    """Pearson Correlation Coefficient"""
    # Handle cases with zero variance to avoid divide by zero
    if np.std(ground_truth) == 0 or np.std(predictions) == 0:
        return 0
    return np.corrcoef(ground_truth, predictions)[0, 1]

def CCC(ground_truth, predictions):
    """Concordance Correlation Coefficient"""
    mean_pred = np.mean(predictions)
    mean_gt = np.mean(ground_truth)
    std_pred = np.std(predictions)
    std_gt = np.std(ground_truth)
    pearson = PCC(ground_truth, predictions)
    
    numerator = 2.0 * pearson * std_pred * std_gt
    denominator = std_pred**2 + std_gt**2 + (mean_pred - mean_gt)**2
    
    if denominator == 0:
        return 0
    return numerator / denominator

# -----------------------------------------------------------------------------
# 2. CONFIGURATION
# -----------------------------------------------------------------------------
DATASET_ROOT = "/kaggle/input/facial-emotion-recognition/images"
MODEL_FP32_PATH = "emonet_8.onnx"
MODEL_INT8_PATH = "emonet_8_int8.onnx"

# Filename to Class Index Map
FILENAME_MAP = {
    "Neutral": 0, "Happy": 1, "Sad": 2, "Surprised": 3,
    "Fear": 4, "Disgust": 5, "Anger": 6, "Contempt": 7
}

# -----------------------------------------------------------------------------
# 3. PREPROCESSING (Center Crop)
# -----------------------------------------------------------------------------
def preprocess_center_crop(image_path):
    try:
        image = cv2.imread(image_path)
        if image is None: return None
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        h, w, _ = image.shape
        min_dim = min(h, w)
        top = (h - min_dim) // 2
        left = (w - min_dim) // 2
        
        # Crop to square
        image = image[top : top+min_dim, left : left+min_dim]
        
        # Resize to 256x256
        image = cv2.resize(image, (256, 256))
        
        # Normalize to [0, 1] and CHW layout
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        
        return np.expand_dims(image, axis=0)
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None

# -----------------------------------------------------------------------------
# 4. LOAD MODELS
# -----------------------------------------------------------------------------
if not os.path.exists(MODEL_FP32_PATH) or not os.path.exists(MODEL_INT8_PATH):
    print("Error: ONNX models not found. Make sure 'emonet_8.onnx' and 'emonet_8_int8.onnx' are in the current directory.")
    exit(1)

print("Loading ONNX sessions...")
# Set providers to CPU as simple quantization usually runs on CPU
session_fp32 = ort.InferenceSession(MODEL_FP32_PATH, providers=['CPUExecutionProvider'])
session_int8 = ort.InferenceSession(MODEL_INT8_PATH, providers=['CPUExecutionProvider'])

# -----------------------------------------------------------------------------
# 5. EVALUATION LOOP
# -----------------------------------------------------------------------------
gt_labels = []

fp32_results = {'emo': [], 'val': [], 'aro': []}
int8_results = {'emo': [], 'val': [], 'aro': []}

folders = [str(i) for i in range(19)] # Folders "0" to "18"

print(f"Starting comparison on {DATASET_ROOT}...")

for folder_name in tqdm(folders, desc="Processing Folders"):
    folder_path = os.path.join(DATASET_ROOT, folder_name)
    if not os.path.isdir(folder_path): continue
        
    for emotion_name, label_idx in FILENAME_MAP.items():
        filename = f"{emotion_name}.jpg"
        image_path = os.path.join(folder_path, filename)
        
        input_tensor = preprocess_center_crop(image_path)
        if input_tensor is None: continue
            
        # --- FP32 Inference ---
        # Outputs: heatmap, expression, valence, arousal (indices 1, 2, 3)
        outs_fp32 = session_fp32.run(None, {"input": input_tensor})
        emo_fp32 = np.argmax(outs_fp32[1]) # Expression logits
        val_fp32 = outs_fp32[2].item()
        aro_fp32 = outs_fp32[3].item()
        
        # --- INT8 Inference ---
        outs_int8 = session_int8.run(None, {"input": input_tensor})
        emo_int8 = np.argmax(outs_int8[1])
        val_int8 = outs_int8[2].item()
        aro_int8 = outs_int8[3].item()
        
        # --- Store Data ---
        gt_labels.append(label_idx)
        
        fp32_results['emo'].append(emo_fp32)
        fp32_results['val'].append(val_fp32)
        fp32_results['aro'].append(aro_fp32)
        
        int8_results['emo'].append(emo_int8)
        int8_results['val'].append(val_int8)
        int8_results['aro'].append(aro_int8)

# -----------------------------------------------------------------------------
# 6. RESULTS ANALYSIS
# -----------------------------------------------------------------------------
if len(gt_labels) == 0:
    print("No images processed.")
    exit()

# Convert to numpy
gt_labels = np.array(gt_labels)
fp32_results['val'] = np.array(fp32_results['val'])
fp32_results['aro'] = np.array(fp32_results['aro'])
int8_results['val'] = np.array(int8_results['val'])
int8_results['aro'] = np.array(int8_results['aro'])

acc_fp32 = accuracy_score(gt_labels, fp32_results['emo'])
acc_int8 = accuracy_score(gt_labels, int8_results['emo'])

# Calculate Fidelity (How close is INT8 to FP32?)
ccc_val = CCC(fp32_results['val'], int8_results['val'])
ccc_aro = CCC(fp32_results['aro'], int8_results['aro'])

print("\n" + "="*50)
print(" ONNX QUANTIZATION COMPARISON RESULTS")
print("="*50)
print(f"Total Images: {len(gt_labels)}")
print("-" * 30)
print(f"1. Emotion Classification Accuracy (Higher is better):")
print(f"   FP32 (Original):  {acc_fp32:.4f} ({acc_fp32*100:.2f}%)")
print(f"   INT8 (Quantized): {acc_int8:.4f} ({acc_int8*100:.2f}%)")
print(f"   Drop in Accuracy: {(acc_fp32 - acc_int8)*100:.2f}%")
print("-" * 30)
print(f"2. Quantization Fidelity (FP32 vs INT8)")
print(f"   (Measures how much precision was lost in regression outputs)")
print(f"   Valence CCC: {ccc_val:.4f} (Max 1.0)")
print(f"   Arousal CCC: {ccc_aro:.4f} (Max 1.0)")
print("="*50)

Loading ONNX sessions...
Starting comparison on /kaggle/input/facial-emotion-recognition/images...


Processing Folders: 100%|██████████| 19/19 [01:17<00:00,  4.06s/it]


 ONNX QUANTIZATION COMPARISON RESULTS
Total Images: 152
------------------------------
1. Emotion Classification Accuracy (Higher is better):
   FP32 (Original):  0.5724 (57.24%)
   INT8 (Quantized): 0.5789 (57.89%)
   Drop in Accuracy: -0.66%
------------------------------
2. Quantization Fidelity (FP32 vs INT8)
   (Measures how much precision was lost in regression outputs)
   Valence CCC: 0.9684 (Max 1.0)
   Arousal CCC: 0.9750 (Max 1.0)
